## Extracting the light chain sequences and saving them into a fasta file ##


In [ ]:
from Bio import SeqIO
import pandas as pd

#loading files
file_csv = "../../documentation/clean_entity_mapping.csv"
chain_type = "../../documentation/pdb_sequences.fasta"

#function to load the csv and to pair pdb to the light subclass
def get_chain_df(file_csv, chain_type):
    df = pd.read_csv(file_csv)
    chain_column = "light_subclass"

    chains = set()

    #normalizing pdb for them all to be in lowercase, light_subclass is converted to uppercase 
    for _, row in df.iterrows():
        pdb = str(row['pdb']).strip().lower()
        subclass = str(row[chain_column]).strip().strip('"').strip("'").upper()

        if not subclass or subclass in ["NAN", "NONE"]:
            continue
        #handling chains with multiple ID, which are separated by a comma 
        for chain in subclass.split(','):
            chain = chain.strip().upper()
            if chain:  
                chains.add((pdb, chain))

    print(f"how many light chains are in the file: {len(chains)}")
    return chains

#using headers from fasta file and assosiating them with pdb ids
def extract_pdb_and_chains(header):
    #spliting by underscore to extract parts + converting to lowercase
    parts = header.split("_")
    pdb_id = parts[0].lower()
    chain_part = next((p for p in parts if p.startswith("chain")), None)
    #extracting the chain part
    if chain_part:
        chain_str = chain_part[len("chain"):].split("_")[0]
        chains = [c.upper() for c in chain_str.split(",")]
        return pdb_id, chains
    return pdb_id, []

#filtering the sequences, keeping only the onces that are in fasta file and clean_entity_mapping.csv
def extract_unique_chain_sequences(fasta_path, chain_mapping, output_path):
    #ensuring that sequences appear only once
    count_total = 0
    unique_sequences = {}
    
    for record in SeqIO.parse(fasta_path, "fasta"):
        header = record.id
        sequence = str(record.seq)
        count_total += 1

        pdb_id, chain_ids = extract_pdb_and_chains(header)
        if not chain_ids:
            continue

        if any((pdb_id, chain) in chain_mapping for chain in chain_ids):
            if sequence not in unique_sequences:
                unique_sequences[sequence] = header

    #writing the fasta file 
    with open(output_path, "w") as out_fasta:
        for seq, header in unique_sequences.items():
            out_fasta.write(f">{header}\n{seq}\n")

    print(f"from {count_total} ")


OUTPUT_FASTA = f"{chain_type.lower()}_sequences_deduplicated.fasta"

chain_mapping = get_chain_df(file_csv, chain_type)
extract_unique_chain_sequences(chain_type, chain_mapping, OUTPUT_FASTA)


how many light chains are in the file: 1406
from 4318 
0 unique sequences
